# Subagents Pattern (하위 에이전트 위임 패턴)

**슈퍼바이저 패턴(Supervisor Pattern)** 은 중앙 감독 에이전트가 여러 전문 하위 에이전트를 조율하는 멀티 에이전트 아키텍처입니다.

이 패턴은 서로 다른 전문 지식이 필요한 작업을 수행할 때 특히 효과적입니다.
각 도메인별로 전문 에이전트를 만들고, 슈퍼바이저가 전체 워크플로우를 이해하고 관리합니다.

**참고**: [LangChain 공식 문서 - Subagents Personal Assistant](https://docs.langchain.com/oss/python/langchain/multi-agent/subagents-personal-assistant)

## 학습 내용
- 하위 에이전트(Sub-agents) 생성
- 에이전트를 도구로 래핑
- 슈퍼바이저 에이전트 구성
- 복합 다중 도메인 요청 처리

## 1. 저수준 API 도구 정의

실제 애플리케이션에서는 Google Calendar, Gmail API 등을 호출하는 도구입니다.
여기서는 예제를 위해 스텁(stub) 구현을 사용합니다.

In [ ]:
def create_calendar_event(
    # 실제로는 Google Calendar API, Outlook API 등을 호출합니다.
def send_email(
    # 실제로는 SendGrid, Gmail API 등을 호출합니다.
def get_available_time_slots(
    # 실제로는 Google Calendar 등의 캘린더 API를 조회합니다.

## 2. 전문 하위 에이전트 생성

각 도메인별로 전문 에이전트를 생성합니다.

### 2.1 캘린더 에이전트

In [ ]:
# 캘린더 에이전트 테스트

### 2.2 이메일 에이전트

In [ ]:
# 이메일 에이전트 테스트

## 3. 하위 에이전트를 도구로 래핑

슈퍼바이저가 하위 에이전트를 호출할 수 있도록 도구로 래핑합니다.
응답 추출은 메시지 타입에 따라 .content 또는 .text를 사용해 호환되게 처리합니다.

In [ ]:
def _last_message_text(messages: list) -> str:
    # 메시지가 비어있으면 빈 문자열 반환
    # 마지막 메시지 추출
    # .text: content가 문자열이든 블록 리스트(Responses API)든 텍스트만 합쳐서 반환
def schedule_event(request: str) -> str:
    # calendar_agent 호출
    # Agent 결과 메시지에서 마지막 텍스트만 추출
def manage_email(request: str) -> str:
    # email_agent 호출
    # Agent 결과 메시지에서 마지막 텍스트만 추출

## 4. 슈퍼바이저 에이전트 생성

여러 하위 에이전트를 조율하는 슈퍼바이저를 생성합니다.

## 5. 슈퍼바이저 사용 예제

### 5.1 단일 도메인 요청

In [ ]:
# 일정 관리만 필요한 경우

### 5.2 복합 다중 도메인 요청

In [ ]:
# 일정 관리 + 이메일 전송이 모두 필요한 경우

-----------------------------------------------------------------------
## 6. 아키텍처 이해

이 시스템은 **3계층 구조**로 이루어져 있습니다:

### 계층 구조

1. **하위 계층 (Bottom Layer)**
   - 정확한 형식을 요구하는 고정형 API 도구
   - 예: `create_calendar_event`, `send_email`
   - 실제 외부 서비스(API)와 직접 상호작용

2. **중간 계층 (Middle Layer)**
   - 하위 에이전트(Sub-Agents)
   - 자연어 입력 → 구조화된 API 호출 → 자연어 응답
   - 예: `calendar_agent`, `email_agent`

3. **상위 계층 (Top Layer)**
   - 슈퍼바이저(Supervisor Agent)
   - 하위 에이전트들을 조율
   - 고수준 명령 인식 및 라우팅
   - 여러 하위 작업 결과 통합

## 7. 슈퍼바이저 패턴의 장점

1. **전문화**: 각 에이전트가 자신의 도메인에 집중
2. **유지보수성**: 도메인별로 독립적으로 개선 가능
3. **확장성**: 새로운 도메인 에이전트를 쉽게 추가
4. **명확성**: 각 계층의 책임이 명확히 분리